# 04 - Recodage des Colonnes & Feature Engineering

Transformation des variables catégorielles + gestion des valeurs aberrantes.

**Pourquoi ?** La régression linéaire ne peut travailler qu'avec des **nombres**.
Les colonnes `room_type`, `neighbourhood_cleansed`, `property_type` sont du texte → on les convertit.

**3 techniques :**
1. **Encodage ordinal** : entier selon un ordre logique (`room_type`)
2. **Encodage fréquentiel** : remplacer par la fréquence d'apparition (quartier, type propriété)
3. **Suppression outliers** : éliminer les prix extrêmes (percentiles 1%–99%)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
os.makedirs('../reports/figures', exist_ok=True)

CITIES = {'Lyon':'lyon','Paris':'paris','Bordeaux':'bordeaux'}
COLORS = {'Lyon':'steelblue','Paris':'coral','Bordeaux':'mediumseagreen'}
dfs = {n: pd.read_csv(f'../data/processed/{k}_clean.csv') for n, k in CITIES.items()}

## 5a. Encodage ordinal du type de logement

On attribue un entier selon le prix typiquement observé :

| room_type | code | Raisonnement |
|-----------|------|--------------|
| Shared room | 0 | Le moins cher |
| Private room | 1 | Chambre privée dans logement partagé |
| Hotel room | 2 | Chambre d'hôtel |
| Entire home/apt | 3 | Le plus cher |

**Pourquoi pas one-hot ?** Avec 4 valeurs et un ordre logique, l'encodage ordinal
est plus simple et permet au modèle de capturer la relation croissante type → prix.

In [ ]:
ROOM_MAP = {'Entire home/apt':3,'Hotel room':2,'Private room':1,'Shared room':0}

for name, df in dfs.items():
    # map() remplace chaque valeur texte par son code entier
    df['room_type_code'] = df['room_type'].map(ROOM_MAP).fillna(0).astype(int)

display(dfs['Lyon'][['room_type','room_type_code']].drop_duplicates()
    .sort_values('room_type_code', ascending=False))

## 5b. Encodage fréquentiel du quartier

**Problème :** des dizaines de quartiers différents par ville → le one-hot créerait trop de colonnes.

**Solution :** remplacer le nom par sa **fréquence relative** dans le dataset.
- Quartier dans 12% des annonces → `neighbourhood_freq = 0.12`
- Quartier dans 3% des annonces → `neighbourhood_freq = 0.03`

**Hypothèse :** quartier très représenté = marché actif = tendance à des prix plus élevés.

`value_counts(normalize=True)` : compte et divise par le total → proportions (somme = 1.0).

In [ ]:
for name, df in dfs.items():
    freq = df['neighbourhood_cleansed'].value_counts(normalize=True)
    df['neighbourhood_freq'] = df['neighbourhood_cleansed'].map(freq)

print('Top 5 quartiers Lyon :')
display(dfs['Lyon'][['neighbourhood_cleansed','neighbourhood_freq']]
    .drop_duplicates().nlargest(5,'neighbourhood_freq'))

## 5c. Encodage fréquentiel du type de propriété

`property_type` est encore plus granulaire que `room_type` :
ex: `"Entire rental unit"`, `"Private room in condo"`, `"Entire loft"`...

Même logique : les types les plus communs correspondent aux prix standards du marché.

In [ ]:
for name, df in dfs.items():
    freq = df['property_type'].value_counts(normalize=True)
    df['property_type_freq'] = df['property_type'].map(freq)

print('Types de propriétés (Lyon) :')
display(dfs['Lyon']['property_type'].value_counts().head(8).to_frame())

## 6. Gestion des valeurs aberrantes (outliers)

**Problème :** un logement à 1€ ou 9999€/nuit fausserait la droite de régression.

**Méthode : percentiles 1% – 99%**
- `quantile(0.01)` : seuil bas (ex: 25€ à Lyon)
- `quantile(0.99)` : seuil haut (ex: 320€ à Lyon)
→ Tout ce qui est en dehors est supprimé.

`reset_index(drop=True)` : après suppression, réindexe de 0 à N-1.

In [ ]:
def remove_outliers(df, col='price', low=0.01, high=0.99):
    """Supprime les lignes hors des percentiles low et high."""
    q_low, q_high = df[col].quantile(low), df[col].quantile(high)
    n = len(df)
    df = df[(df[col] >= q_low) & (df[col] <= q_high)].copy()
    print(f'  Seuils [{q_low:.0f}€ – {q_high:.0f}€] | Supprimés : {n-len(df)} lignes')
    return df.reset_index(drop=True)

for name in list(dfs.keys()):
    print(f'{name} :')
    dfs[name] = remove_outliers(dfs[name])

## Boxplot après nettoyage

Le **boxplot** (boîte à moustaches) :
- **Boîte** : Q1 à Q3 (50% des données)
- **Trait central** : médiane
- **Moustaches** : 1.5 × IQR au-delà de Q1 et Q3
- **Points au-delà** : outliers résiduels

Après nettoyage, on ne doit plus voir de valeurs très éloignées de la boîte.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, df) in zip(axes, dfs.items()):
    ax.boxplot(df['price'], vert=False, patch_artist=True,
               boxprops=dict(facecolor=COLORS[name], alpha=0.7))
    ax.set_title(f'Boxplot prix après nettoyage - {name}')
    ax.set_xlabel('Prix (€)')
plt.tight_layout()
plt.savefig('../reports/figures/boxplot_prix.png', bbox_inches='tight')
plt.show()

## Sauvegarde avec les nouvelles features

On sauvegarde dans `_features.csv` : ces fichiers contiennent les 3 nouvelles colonnes
`room_type_code`, `neighbourhood_freq`, `property_type_freq`.
Le notebook 05 utilisera ces fichiers directement.

In [ ]:
for name, key in CITIES.items():
    dfs[name].to_csv(f'../data/processed/{key}_features.csv', index=False)
    print(f'{name} -> {dfs[name].shape}')